# 📦 ETL Bronze - Ingesta de Datos Climáticos

## 🧭 Descripción General

Este proceso ETL en la capa **Bronze** tiene como objetivo la **ingesta de datos crudos (raw)** desde la API de clima (*Weather API*), almacenándolos en formato JSON sin transformaciones.

La información recolectada corresponde a múltiples ciudades y distintos tipos de datos climáticos.

---

## 🎯 Objetivo

* Extraer datos desde una API externa
* Almacenar datos en formato **raw (sin procesar)**
* Organizar la información en una estructura tipo **Data Lake**
* Preparar los datos para futuras transformaciones (Silver)

---

## 🌍 Fuente de Datos

* API: `http://api.weatherapi.com/v1`
* Autenticación: API Key

### Endpoints utilizados:

| Endpoint  | Descripción                     |
| --------- | ------------------------------- |
| forecast  | Pronóstico a 3 días             |
| history   | Datos históricos del día actual |
| astronomy | Información astronómica         |

---

## 🏙️ Ciudades Procesadas

* Venado Tuerto
* Rosario
* Firmat
* Rafaela
* Casilda
* Cañada de Gómez
* San Lorenzo
* El Trébol
* San Justo

---

## ⏱️ Manejo de Fechas

* `date_str`: Fecha en formato `YYYY-MM-DD`
* `timestamp`: Marca temporal en formato UTC

Ejemplo:

```
2026-03-26T00-23-17Z
```

---

## 🧱 Estructura de Almacenamiento

Ruta base:

```
/Volumes/workspace/default/bronce_clima/
```

Estructura generada:

```
bronce_clima/
│
├── forecast/
│   └── city=rosario/
│       └── date=2026-03-26/
│           └── 2026-03-26T00-23-17Z.json
│
├── history/
│   └── city=rosario/
│       └── date=2026-03-26/
│           └── archivo.json
│
└── astronomy/
    └── city=rosario/
        └── date=2026-03-26/
            └── archivo.json
```

---

## ⚙️ Lógica del Proceso

1. Se define una lista de ciudades
2. Se generan URLs dinámicas por endpoint
3. Se realiza una llamada HTTP por ciudad y endpoint
4. Se valida la respuesta (`status_code == 200`)
5. Se guarda el JSON en el filesystem


## ⚠️ Consideraciones Técnicas

### ❗ Escritura de archivos

Actualmente se utiliza:

* `os.makedirs`
* `open()`

Esto implica que los datos pueden guardarse en el **files**


In [0]:
import requests
import json
import logging
import time
import unicodedata
from datetime import datetime

In [0]:
api_key = dbutils.widgets.get("api_key")
base_url = "http://api.weatherapi.com/v1"
path_raiz = "/Volumes/workspace/default/bronce_clima"

# LOGGING
logging.basicConfig(level=logging.INFO)

def normalizar_texto(texto):
    # 1. Pasar a minúsculas
    texto = texto.lower()
    # 2. Reemplazar espacios por guiones bajos ANTES de normalizar
    texto = texto.replace(" ", "_")
    # 3. Descomponer caracteres (NFD)
    texto = unicodedata.normalize("NFD", texto)
    # 4. Eliminar tildes y caracteres especiales filtrando por ASCII
    texto = "".join([c for c in texto if unicodedata.category(c) != 'Mn'])
    # 5. Codificar y decodificar para limpiar cualquier rastro no-ascii
    texto = texto.encode("ascii", "ignore").decode("utf-8")
    return texto


now = datetime.utcnow()
date_str = now.strftime("%Y-%m-%d")
timestamp = now.strftime("%Y-%m-%dT%H-%M-%SZ")

# ENDPOINTS
endpoints = {
    "forecast": lambda ciudad: f"{base_url}/forecast.json?key={api_key}&q={ciudad}&days=3&aqi=yes&alerts=yes",
    "history": lambda ciudad: f"{base_url}/history.json?key={api_key}&q={ciudad}&dt={date_str}",
    "astronomy": lambda ciudad: f"{base_url}/astronomy.json?key={api_key}&q={ciudad}"
}



ciudades = [
    "Venado Tuerto", "Rosario", "Firmat", "Rafaela",
    "Casilda", "Cañada de Gomez", "San Lorenzo",
    "El Trebol", "San Justo","Barrancas"
]

for ciudad in ciudades:
    city_folder = normalizar_texto(ciudad)

    for endpoint_name, url_func in endpoints.items():
        url = url_func(ciudad)

        try:
            response = requests.get(url)
            # VALIDACIÓN RESPUEST
            if response.status_code == 200:
                raw_data = response.json()

                if "error" in raw_data:
                    logging.error(f"API error en {endpoint_name} - {ciudad}: {raw_data}")
                    continue
                # METADATA
                enriched_data = {
                    "data": raw_data,
                    "metadata": {
                        "ciudad": ciudad,
                        "endpoint": endpoint_name,
                        "ingestion_time": timestamp,
                        "source": "weatherapi"
                    }
                }

                # PATH
                full_folder_path = f"{path_raiz}/{endpoint_name}/city={city_folder}/date={date_str}"
                file_name = f"{timestamp}.json"
                full_file_path = f"{full_folder_path}/{file_name}"

                # ESCRITURA EN VOLUMEN
                dbutils.fs.put(
                    full_file_path,
                    json.dumps(enriched_data, ensure_ascii=False),
                    overwrite=True
                )

                logging.info(f"Guardado OK: {endpoint_name} | {city_folder}")

            else:
                # ERROR API
                error_info = {
                    "ciudad": ciudad,
                    "endpoint": endpoint_name,
                    "status_code": response.status_code,
                    "timestamp": timestamp
                }

                error_path = f"{path_raiz}/_errors/{timestamp}_{city_folder}_{endpoint_name}.json"

                dbutils.fs.put(
                    error_path,
                    json.dumps(error_info),
                    overwrite=True
                )

                logging.error(f"Error {response.status_code} en {endpoint_name} - {ciudad}")

        except Exception as e:
            # ERROR CRÍTICO
            error_info = {
                "ciudad": ciudad,
                "endpoint": endpoint_name,
                "error": str(e),
                "timestamp": timestamp
            }

            error_path = f"{path_raiz}/_errors/{timestamp}_{city_folder}_{endpoint_name}_exception.json"

            dbutils.fs.put(
                error_path,
                json.dumps(error_info),
                overwrite=True
            )

            logging.exception(f"Fallo crítico en {endpoint_name} - {ciudad}")

        # RATE LIMIT
        time.sleep(1)